In [5]:
# Renombrar la las caperta que tenga un numero de legajo para que el nombre sea solo el legajo.
import os
import re

def renombrar_carpeta(ruta_carpeta):
    nombre_actual = os.path.basename(ruta_carpeta)

    # Extraer el número de legajo (5 dígitos) del nombre de la carpeta
    match = re.search(r'\b(\d{5})\b', nombre_actual)
    if not match:
        return

    legajo = match.group(1)
    ruta_padre = os.path.dirname(ruta_carpeta)
    nueva_ruta = os.path.join(ruta_padre, legajo)

    if ruta_carpeta == nueva_ruta:
        return

    if os.path.exists(nueva_ruta):
        print(f"No se renombró '{ruta_carpeta}' porque ya existe '{nueva_ruta}'")
        return

    os.rename(ruta_carpeta, nueva_ruta)
    print(f"Carpeta '{ruta_carpeta}' renombrada a '{nueva_ruta}'")



def renombrar_subcarpetas(ruta_base="."):
    # topdown=False para renombrar primero las carpetas más internas
    for raiz, dirs, _ in os.walk(ruta_base, topdown=False):
        for nombre_carpeta in dirs:
            ruta_carpeta = os.path.join(raiz, nombre_carpeta)
            renombrar_carpeta(ruta_carpeta)



# Ejemplo de uso: recorre todas las subcarpetas desde la ruta indicada
renombrar_subcarpetas("/Users/adibattista/Documents/GitHub/tup26-p3/practicos")
renombrar_subcarpetas("/Users/adibattista/Documents/GitHub/tup26-p3/practicos1")
renombrar_subcarpetas("/Users/adibattista/Documents/GitHub/tup26-p3/practicos2")


Carpeta '/Users/adibattista/Documents/GitHub/tup26-p3/practicos/61641 - Figueroa, Nahuel Ramon Al' renombrada a '/Users/adibattista/Documents/GitHub/tup26-p3/practicos/61641'
Carpeta '/Users/adibattista/Documents/GitHub/tup26-p3/practicos/63354 - Perondi, Luciano' renombrada a '/Users/adibattista/Documents/GitHub/tup26-p3/practicos/63354'
Carpeta '/Users/adibattista/Documents/GitHub/tup26-p3/practicos/63461 - Cativa, Facundo Simón' renombrada a '/Users/adibattista/Documents/GitHub/tup26-p3/practicos/63461'
Carpeta '/Users/adibattista/Documents/GitHub/tup26-p3/practicos/63647 - Paz, Valentina' renombrada a '/Users/adibattista/Documents/GitHub/tup26-p3/practicos/63647'
Carpeta '/Users/adibattista/Documents/GitHub/tup26-p3/practicos/63399 - Lazarte, Gonzalo Romeo' renombrada a '/Users/adibattista/Documents/GitHub/tup26-p3/practicos/63399'
Carpeta '/Users/adibattista/Documents/GitHub/tup26-p3/practicos/61801 - Benega, Maximiliano Abrah' renombrada a '/Users/adibattista/Documents/GitHub/tup

In [69]:
from pathlib import Path
import re

RAIZ = Path("/Users/adibattista/Documents/GitHub/tup26-p3")
DESTINO_BASE = RAIZ / "practicos"
ORIGENES = [RAIZ / "practicos1", RAIZ / "practicos2"]
MAX_LINEAS_DESTINO = 10
MIN_LINEAS_ORIGEN = 10
MODO_SIMULACION = False  # Cambiar a False para copiar de verdad

PATRON_LEGAJO = re.compile(r"\b(\d{5})\b")

def rel(camino: str | Path) -> str:
    ruta = Path(camino)
    try:
        return f"{ruta.relative_to(RAIZ).as_posix()}"
    except ValueError:
        return ruta.as_posix()

def extraer_legajo_desde_path(path: Path) -> str | None:
    for parte in path.parts:
        match = PATRON_LEGAJO.search(parte)
        if match:
            return match.group(1)
    return None

def contar_lineas(path: Path) -> int:
    if not path.exists():
        return 0
    return len(path.read_text(encoding="utf-8", errors="replace").splitlines())

def leer_texto(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="replace")

def archivos_son_identicos(path1: str | Path, path2: str | Path) -> bool:
    return leer_texto(Path(path1)) == leer_texto(Path(path2))

def describir_archivo(path: str | Path, lineas: int | None = None) -> str:
    ruta = Path(path)
    cantidad_lineas = contar_lineas(ruta) if lineas is None else lineas
    return f"({cantidad_lineas:3}) {rel(ruta):50} {ruta}"

def agrupar_fuentes_por_contenido(candidatas: list[Path]) -> dict[str, list[Path]]:
    textos: dict[str, list[Path]] = {}
    for path in candidatas:
        textos.setdefault(leer_texto(path), []).append(path)
    return textos

def filtrar_fuentes_distintas_a_destino(
    candidatas: list[Path],
    destino: Path,
    ) -> list[Path]:
    destino_texto = leer_texto(destino)
    fuentes_distintas: list[Path] = []
    for paths in agrupar_fuentes_por_contenido(candidatas).values():
        if leer_texto(paths[0]) == destino_texto:
            continue
        fuentes_distintas.append(paths[0])
    return fuentes_distintas

def recolectar_fuentes(
    origenes: list[Path],
    min_lineas_origen: int = 10,
) -> dict[str, list[Path]]:
    fuentes: dict[str, list[Path]] = {}
    for origen in origenes:
        for path in origen.rglob("sortx.cs"):
            if contar_lineas(path) < min_lineas_origen:
                continue
            legajo = extraer_legajo_desde_path(path)
            if not legajo:
                continue
            fuentes.setdefault(legajo, []).append(path)
    return fuentes

def elegir_fuente(candidatas: list[Path]) -> tuple[Path | None, str | None]:
    if not candidatas:
        return None, "sin_fuente"
    textos = agrupar_fuentes_por_contenido(candidatas)
    if len(textos) == 1:
        return candidatas[0], None
    return None, "fuentes_multiples"

def sincronizar_sortx(
    destino_base: Path,
    origenes: list[Path],
    max_lineas_destino: int = 10,
    min_lineas_origen: int = 10,
    modo_simulacion: bool = True,
):
    fuentes_por_legajo = recolectar_fuentes(
        origenes,
        min_lineas_origen=min_lineas_origen,
    )
    copiados = []
    conflictos = []
    sin_fuente = []
    conflictos_fuente = []
    for destino in destino_base.rglob("sortx.cs"):
        legajo = extraer_legajo_desde_path(destino)
        if not legajo:
            continue

        lineas_destino = contar_lineas(destino)
        candidatas = fuentes_por_legajo.get(legajo, [])
        if not candidatas:
            sin_fuente.append({
                "legajo": legajo,
                "destino": str(destino),
                "lineas_destino": lineas_destino,
            })
            continue

        candidatas = filtrar_fuentes_distintas_a_destino(candidatas, destino)
        if not candidatas:
            continue

        fuente, motivo = elegir_fuente(candidatas)
        if motivo == "fuentes_multiples":
            conflictos_fuente.append({
                "legajo": legajo,
                "destino": str(destino),
                "lineas_destino": lineas_destino,
                "fuentes": [
                    {
                        "ruta": str(path),
                        "lineas": contar_lineas(path),
                    }
                    for path in candidatas
                ],
            })
            continue

        lineas_fuente = contar_lineas(fuente)
        es_identico = archivos_son_identicos(fuente, destino)
        if lineas_destino >= max_lineas_destino:
            if es_identico:
                continue
            conflictos.append({
                "legajo": legajo,
                "destino": str(destino),
                "lineas_destino": lineas_destino,
                "fuente": str(fuente),
                "lineas_fuente": lineas_fuente,
                "es_identico": es_identico,
            })
            continue

        if not modo_simulacion:
            destino.write_text(leer_texto(fuente), encoding="utf-8")

        copiados.append({
            "legajo": legajo,
            "destino": str(destino),
            "lineas_destino": lineas_destino,
            "fuente": str(fuente),
            "lineas_fuente": lineas_fuente,
            "accion": "simulado" if modo_simulacion else "copiado",
        })

    print(f"Copiados/simulados: {len(copiados)}")
    print(f"Conflictos por destino con >= {max_lineas_destino} líneas: {len(conflictos)}")
    print(f"Conflictos por múltiples fuentes distintas: {len(conflictos_fuente)}")
    print(f"Sin fuente encontrada: {len(sin_fuente)}")

    if copiados:
        print("\n=== Copiados o simulados ===")
        for item in copiados:
            print(
                f"[{item['accion']}] {item['legajo']}\n"
                f"  destino: {describir_archivo(item['destino'], item['lineas_destino'])}\n"
                f"  fuente : {describir_archivo(item['fuente'], item['lineas_fuente'])}"
            )

    if conflictos:
        print("\n=== Conflictos por destino no reemplazado ===")
        for item in conflictos:
            marca = " 🟢" if item["es_identico"] else ""
            print(
                f"[conflicto] {item['legajo']}{marca}\n"
                f"  destino: {describir_archivo(item['destino'], item['lineas_destino'])}\n"
                f"  fuente : {describir_archivo(item['fuente'], item['lineas_fuente'])}"
            )

    if conflictos_fuente:
        print("\n=== Conflictos por múltiples fuentes ===")
        for item in conflictos_fuente:
            print(f"[conflicto-fuentes] {item['legajo']}")
            print(f"  destino: {describir_archivo(item['destino'], item['lineas_destino'])}")
            for fuente in item["fuentes"]:
                print(f"  fuente : {describir_archivo(fuente['ruta'], fuente['lineas'])}")

    if sin_fuente:
        print("\n=== Destinos sin fuente ===")
        for indice, item in enumerate(sin_fuente, start=1):
            print(
                f"{indice:2}) [sin-fuente] {item['legajo']} -> "
                f"{describir_archivo(item['destino'], item['lineas_destino'])}"
            )
    
    return {
        "copiados": copiados,
        "conflictos": conflictos,
        "conflictos_fuente": conflictos_fuente,
        "sin_fuente": sin_fuente,
    }

resultado = sincronizar_sortx(
    destino_base=DESTINO_BASE,
    origenes=ORIGENES,
    max_lineas_destino=MAX_LINEAS_DESTINO,
    min_lineas_origen=MIN_LINEAS_ORIGEN,
    modo_simulacion=MODO_SIMULACION,
)


Copiados/simulados: 0
Conflictos por destino con >= 10 líneas: 0
Conflictos por múltiples fuentes distintas: 0
Sin fuente encontrada: 29

=== Destinos sin fuente ===
 1) [sin-fuente] 61641 -> (  7) practicos/61641/TP1/sortx.cs                       /Users/adibattista/Documents/GitHub/tup26-p3/practicos/61641/TP1/sortx.cs
 2) [sin-fuente] 61489 -> (214) practicos/61489/TP1/sortx.cs                       /Users/adibattista/Documents/GitHub/tup26-p3/practicos/61489/TP1/sortx.cs
 3) [sin-fuente] 61026 -> (204) practicos/61026/TP1/sortx.cs                       /Users/adibattista/Documents/GitHub/tup26-p3/practicos/61026/TP1/sortx.cs
 4) [sin-fuente] 63205 -> (  7) practicos/63205/TP1/sortx.cs                       /Users/adibattista/Documents/GitHub/tup26-p3/practicos/63205/TP1/sortx.cs
 5) [sin-fuente] 64016 -> (  7) practicos/64016/TP1/sortx.cs                       /Users/adibattista/Documents/GitHub/tup26-p3/practicos/64016/TP1/sortx.cs
 6) [sin-fuente] 63232 -> (  7) practicos/63232/T

In [71]:
from pathlib import Path

def listar_sortx_en_practicos(destino_base: Path):
    resultados = []
    for destino in sorted(destino_base.rglob("sortx.cs")):
        legajo = extraer_legajo_desde_path(destino)
        if not legajo:
            continue
        resultados.append((legajo, contar_lineas(destino), destino))
    resultados.sort(key=lambda item: item[0])
    print("legajo | lineas | archivo")
    print("-" * 80)
    for legajo, lineas, destino in resultados:
        print(f"{legajo} | {lineas:3} | {rel(destino)}")
    return resultados

listado_sortx = listar_sortx_en_practicos(DESTINO_BASE)


legajo | lineas | archivo
--------------------------------------------------------------------------------
61026 | 204 | practicos/61026/TP1/sortx.cs
61057 |   7 | practicos/61057/TP1/sortx.cs
61161 | 242 | practicos/61161/TP1/sortx.cs
61489 | 214 | practicos/61489/TP1/sortx.cs
61490 | 169 | practicos/61490/TP1/sortx.cs
61577 | 168 | practicos/61577/TP1/sortx.cs
61581 | 397 | practicos/61581/TP1/sortx.cs
61641 |   7 | practicos/61641/TP1/sortx.cs
61801 |   7 | practicos/61801/TP1/sortx.cs
61907 |   7 | practicos/61907/TP1/sortx.cs
62844 | 274 | practicos/62844/TP1/sortx.cs
63137 | 290 | practicos/63137/TP1/sortx.cs
63150 | 202 | practicos/63150/TP1/sortx.cs
63174 | 220 | practicos/63174/TP1/sortx.cs
63182 | 280 | practicos/63182/TP1/sortx.cs
63205 |   7 | practicos/63205/TP1/sortx.cs
63207 | 268 | practicos/63207/TP1/sortx.cs
63208 |  49 | practicos/63208/TP1/sortx.cs
63211 |   7 | practicos/63211/TP1/sortx.cs
63213 | 244 | practicos/63213/TP1/sortx.cs
63216 | 276 | practicos/63216/TP1